# Instalação das Dependências

Instala o Google Chrome no ambiente Linux do Colab e as bibliotecas Python necessárias.

In [1]:
# 1. Adiciona o repositório oficial do Google Chrome e instala
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get update -y
!apt-get install -y google-chrome-stable

# 2. Instala as bibliotecas de Web Scraping para Python
!pip install selenium beautifulsoup4 webdriver-manager -q

OK
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [2,548 B]
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Get:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:9 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,415 B]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:11 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:12 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,162 kB]
Get:14 https://

# A Classe: NarutoArenaScraper



Define a estrutura limpa e orientada a objetos do nosso scraper.

In [2]:
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By


class NarutoArenaScraper:
  """Classe responsável pela navegação automatizada e extração do HTML

  no Naruto-Arena utilizando Selenium Headless.
  """

  def __init__(self, base_url: str = 'https://www.naruto-arena.site/'):
    self.base_url = base_url
    self.driver = self._setup_driver()

  def _setup_driver(self) -> webdriver.Chrome:
    """Configura o Chrome em modo Headless para o ambiente do Colab."""
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
        ' AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0'
        ' Safari/537.36'
    )
    return webdriver.Chrome(options=options)

  def fetch_html(self, path: str = '') -> str:
    """Acessa uma URL do site e retorna o HTML completo após a renderização."""
    url = f'{self.base_url}{path}'.strip('/')
    self.driver.get(url)
    time.sleep(3)  # Aguarda o carregamento dos scripts dinâmicos (JS)
    return self.driver.page_source

  def close(self):
    """Encerra a sessão do navegador."""
    self.driver.quit()

# Teste de Conectividade

Executa o teste de requisição

In [3]:
# Instancia o scraper e realiza o teste
scraper = NarutoArenaScraper()
try:
  html_content = scraper.fetch_html('')
  print(f'Sucesso! HTML capturado com {len(html_content)} caracteres.')
finally:
  scraper.close()

Sucesso! HTML capturado com 134469 caracteres.


# Captura Segura de Credenciais (Interativa)
Rode para preencher suas credenciais de forma segura em memória (sem salvar no código do notebook)

In [5]:
import getpass
import os

# Solicita o login sem salvar em texto no código
NA_USER = getpass.getpass("Digite o E-mail/Usuário do Naruto-Arena: ")
NA_PASS = getpass.getpass("Digite a Senha (o texto ficará oculto): ")

print("Credenciais armazenadas com segurança em memória para a sessão atual!")

Digite o E-mail/Usuário do Naruto-Arena: ··········
Digite a Senha (o texto ficará oculto): ··········
Credenciais armazenadas com segurança em memória para a sessão atual!


# Classe NarutoArenaScraper (Com Método de Login)

Autenticação adicionada.

In [6]:
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By


class NarutoArenaScraper:
  """Classe responsável pela navegação automatizada, autenticação e extração no Naruto-Arena utilizando Selenium Headless."""

  def __init__(self, base_url: str = 'https://www.naruto-arena.site/'):
    self.base_url = base_url.rstrip('/')
    self.driver = self._setup_driver()

  def _setup_driver(self) -> webdriver.Chrome:
    """Configura o Chrome em modo Headless otimizado para o ambiente do Colab."""
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
        ' AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0'
        ' Safari/537.36'
    )
    return webdriver.Chrome(options=options)

  def login(self, username: str, password: str) -> bool:
    """Acessa a página /login, preenche o formulário via IDs e autentica a sessão."""
    try:
      login_url = f'{self.base_url}/login'
      self.driver.get(login_url)
      time.sleep(2)

      # Seletores reais identificados no HTML do site
      user_field = self.driver.find_element(By.ID, 'login-username')
      pass_field = self.driver.find_element(By.ID, 'login-password')

      user_field.clear()
      user_field.send_keys(username)
      pass_field.clear()
      pass_field.send_keys(password)

      # Submete o formulário enviando o comando 'ENTER' no campo de senha
      from selenium.webdriver.common.keys import Keys

      pass_field.send_keys(Keys.RETURN)

      time.sleep(4)
      print('Autenticação processada! Sessão ativa.')
      return True

    except Exception as e:
      print(f'Falha ao tentar realizar o login: {e}')
      return False

  def fetch_authenticated_page(self, path: str) -> str:
    """Navega para qualquer página interna do site reutilizando a sessão logada."""
    target_url = f'{self.base_url}/{path.lstrip("/")}'
    self.driver.get(target_url)
    time.sleep(3)  # Aguarda a renderização completa do JavaScript
    return self.driver.page_source

  def close(self):
    """Encerra a sessão do navegador."""
    self.driver.quit()

# Testando o Login com Credenciais Seguras
Execução no Colab para validar se o login funciona com as variáveis NA_USER e NA_PASS salvas em memória:

In [7]:
# Instancia o scraper e executa o teste de login
scraper = NarutoArenaScraper()

try:
  if scraper.login(NA_USER, NA_PASS):
    # Teste: Acessa a página principal após o login para verificar o HTML autenticado
    authenticated_html = scraper.fetch_authenticated_page('')
    print(
      f'Sucesso! Página capturada com {len(authenticated_html)} caracteres.'
    )
finally:
  scraper.close()

Autenticação processada! Sessão ativa.
Sucesso! Página capturada com 134130 caracteres.


# Classe NarutoArenaParser (BeautifulSoup)
Processar o HTML baixado:

In [ ]:
import re
from bs4 import BeautifulSoup


class NarutoArenaParser:
  """Classe responsável por transformar o HTML das páginas do Naruto-Arena em dados JSON estruturados."""

  @staticmethod
  def extract_character_links(html_content: str, base_url: str) -> list[str]:
    """Extrai todos os links das páginas individuais dos personagens da tela /characters-and-skills."""
    soup = BeautifulSoup(html_content, "html.parser")
    links = []

    # Encontra todas as divs 'pagedescription' que contêm os links dos personagens
    cards = soup.find_all("div", class_="pagedescription")
    for card in cards:
      link_elem = card.find("a", class_="subjlink")
      if link_elem and link_elem.get("href"):
        href = link_elem["href"]
        if not href.startswith("http"):
          href = f"{base_url.rstrip('/')}/{href.lstrip('/')}"
        if href not in links:
          links.append(href)

    return links

  @staticmethod
  def parse_character_page(html_content: str, char_url: str) -> dict:
    """Extrai os dados detalhados do personagem e de todas as suas habilidades a partir do HTML da página individual."""
    soup = BeautifulSoup(html_content, "html.parser")

    # 1. Nome do Personagem
    title_elem = soup.find("h1", class_="home-h1")
    char_name = (
        title_elem.get_text(strip=True)
        if title_elem
        else char_url.split("/")[-1].replace("-", " ").title()
    )

    skills = []

    # 2. Habilidades (<div class="charskill">)
    skill_divs = soup.find_all("div", class_="charskill")
    for s_div in skill_divs:
      # Nome da Habilidade
      h2_elem = s_div.find("h2")
      skill_name = h2_elem.get_text(strip=True) if h2_elem else "Desconhecido"

      # Descrição
      descr_elem = s_div.find("div", class_="skilldescr")
      description = ""
      if descr_elem:
        # Pega todo o texto da descrição (ignorando tags filhas)
        description = descr_elem.get_text(" ", strip=True)

      # Cooldown
      cooldown_text = "None"
      for u_tag in s_div.find_all("u", class_="font-fix"):
        if "Cooldown" in u_tag.get_text():
          # O texto do cooldown fica logo após a tag <u>
          cooldown_text = (
              u_tag.next_sibling.strip() if u_tag.next_sibling else "None"
          )
          break

      # Classes/Categorias (Physical, Instant, Melee, etc.)
      classes = []
      pageinfo = s_div.find("div", class_="pageinfo")
      if pageinfo:
        info_text = pageinfo.get_text(strip=True)
        if "Classes:" in info_text:
          classes_str = info_text.split("Classes:")[-1]
          classes = [c.strip().strip(".") for c in classes_str.split(",")]

      # Imagens do Chakra exigido
      chakra_imgs = [img["src"] for img in s_div.find_all("img", class_="page-energy") if "src" in img.attrs]

      skills.append({
          "name": skill_name,
          "description": description,
          "cooldown": cooldown_text,
          "classes": classes,
          "chakra_imgs": chakra_imgs,  # Salva as URLs das imagens para mapearmos os tipos no Parser de IA
      })

    return {"name": char_name, "url": char_url, "skills": skills}

# Loop de Raspagem e Salvamento do JSON
Agora rodamos o fluxo completo: autentica, pega a lista de links, percorre cada personagem com um pequeno intervalo de pausa para não sobrecarregar o servidor do jogo, e salva o arquivo naruto_arena_characters_raw.json.

In [ ]:
import json
import os
import time

# Executa o fluxo completo
scraper = NarutoArenaScraper()

try:
  print("1. Autenticando...")
  if scraper.login(NA_USER, NA_PASS):
    print("2. Coletando lista de personagens...")
    list_html = scraper.fetch_authenticated_page("characters-and-skills")

    char_links = NarutoArenaParser.extract_character_links(
        list_html, scraper.base_url
    )
    print(f"Total de personagens encontrados: {len(char_links)}")

    all_characters = []

    # Extrai os dados de cada personagem
    for idx, link in enumerate(char_links, 1):
      print(f"[{idx}/{len(char_links)}] Coletando: {link}...")
      char_html = scraper.fetch_authenticated_page(
          link.replace(scraper.base_url, "")
      )
      char_data = NarutoArenaParser.parse_character_page(char_html, link)
      all_characters.append(char_data)
      time.sleep(1)  # Pausa de 1 segundo entre requisições

    # Cria pasta data/raw se não existir
    os.makedirs("../data/raw", exist_ok=True)
    output_path = "../data/raw/naruto_arena_characters_raw.json"

    with open(output_path, "w", encoding="utf-8") as f:
      json.dump(all_characters, f, ensure_ascii=False, indent=2)

    print(
        f"\nSUCESSO TOTAL! {len(all_characters)} personagens foram salvos em: {output_path}"
    )

finally:
  scraper.close()

1. Autenticando...
Autenticação processada! Sessão ativa.
2. Coletando lista de personagens...
Total de personagens encontrados: 219
[1/219] Coletando: https://www.naruto-arena.site/chars/uzumaki-naruto...
[2/219] Coletando: https://www.naruto-arena.site/chars/haruno-sakura...
[3/219] Coletando: https://www.naruto-arena.site/chars/uchiha-sasuke...
[4/219] Coletando: https://www.naruto-arena.site/chars/inuzuka-kiba...
[5/219] Coletando: https://www.naruto-arena.site/chars/aburame-shino...
[6/219] Coletando: https://www.naruto-arena.site/chars/hyuuga-hinata...
[7/219] Coletando: https://www.naruto-arena.site/chars/nara-shikamaru...
[8/219] Coletando: https://www.naruto-arena.site/chars/akimichi-chouji...
[9/219] Coletando: https://www.naruto-arena.site/chars/yamanaka-ino...
[10/219] Coletando: https://www.naruto-arena.site/chars/tenten...
[11/219] Coletando: https://www.naruto-arena.site/chars/hyuuga-neji...
[12/219] Coletando: https://www.naruto-arena.site/chars/rock-lee...
[13/219] Col

# Compreender a Ontologia do game
Ao inspecionar a página do guia no navegador, observamos que o básico do jogo está estruturado dentro do container principal (#content ou .brspacer), dividindo as regras em seções (Chakra/Energia, Classes de Habilidades, Estados de Efeito e Turnos).

In [12]:
import json
import os
from bs4 import BeautifulSoup


class NarutoArenaOntologyParser:
  """Classe responsável por extrair o conteúdo dinâmico da página /the-basics trazida pelo Selenium do site real."""

  @staticmethod
  def parse_basics_page(html_content: str) -> dict:
    soup = BeautifulSoup(html_content, "html.parser")
    container = soup.find("div", id="container2")

    scraped_sections = {}
    current_heading = "General Rules"
    scraped_sections[current_heading] = []

    if container:
      # Pega apenas os elementos de texto diretos para evitar repetição por aninhamento
      elements = container.find_all(
          ["h1", "h2", "h3", "p", "b", "li"], recursive=True
      )

      for elem in elements:
        text = elem.get_text(" ", strip=True)

        if not text or len(text) < 3:
          continue

        # Ignora migalhas de pão (breadcrumbs) e links de navegação
        if text in [
            "Naruto Arena",
            "Game manual",
            "Characters and Skills",
            "The Basics",
        ]:
          continue

        # Cabeçalhos definem uma nova seção
        if elem.name in ["h1", "h2", "h3"]:
          current_heading = text
          if current_heading not in scraped_sections:
            scraped_sections[current_heading] = []
        else:
          # Adiciona o parágrafo ou item de lista se não for duplicado
          if text not in scraped_sections[current_heading]:
            scraped_sections[current_heading].append(text)

    return {
        "source_url": "https://www.naruto-arena.site/the-basics",
        "scraped_at_runtime": True,
        "content_by_sections": scraped_sections,
    }

In [13]:
# Executa a extração e grava no JSON
scraper = NarutoArenaScraper()

try:
  print("1. Acessando /the-basics com a sessão do Selenium...")
  if scraper.login(NA_USER, NA_PASS):
    basics_html = scraper.fetch_authenticated_page("the-basics")

    print("2. Extraindo o texto puro da ontologia...")
    ontology_data = NarutoArenaOntologyParser.parse_basics_page(basics_html)

    # Salva em data/raw/game_ontology_raw.json
    os.makedirs("../data/raw", exist_ok=True)
    ontology_path = "../data/raw/game_ontology_raw.json"

    with open(ontology_path, "w", encoding="utf-8") as f:
      json.dump(ontology_data, f, ensure_ascii=False, indent=2)

    print(
        f"\nSUCESSO! Arquivo salvo em: {os.path.abspath(ontology_path)}"
    )
finally:
  scraper.close()

1. Acessando /the-basics com a sessão do Selenium...
Autenticação processada! Sessão ativa.
2. Extraindo o texto puro da ontologia...

SUCESSO! Arquivo salvo em: /data/raw/game_ontology_raw.json


No menu da esquerda do Colab, clique no ícone de Pasta (Arquivos).

1. Navegue até a pasta data $\to$ raw, clique nos 3 pontinhos ao lado do arquivo naruto_arena_characters_raw.json & game_ontology_raw.json e escolha Fazer download.

2. Abra o seu repositório no navegador: github.com/YOUR-USER/team-combinatorial-optimization-ai.

3. Entre na pasta data/raw/ (se a pasta ainda não existir no GitHub, clique em Add file $\to$ Create new file, digite data/raw/README.md e confirme).

4. Clique em Add file $\to$ Upload files, arraste o arquivo naruto_arena_characters_raw.json e game_ontology_raw.json que você baixou e clique em Commit changes.